# Zadanie 3.2: sieć rezerwuarowa (ESN) uczona na trajektoriach ODE

Cel (issue #21): zbudować i przebadać sieć Echo State Network (`reservoirpy`) uczoną na
trajektoriach surogatu ODE (`ODE_2`), z wejściem $(M(t),\, U_{\Delta t}(t))$ i wyjściem $M(t + \Delta t)$:
1. **ekstrapolacja w czasie w trybie zamkniętym (closed-loop):** trening na $[0, T_{\text{train}}]$,
   a następnie autonomiczne rozwijanie poza $T_{\text{train}}$ (z podawaniem znanego $U(t)$ i sprzężeniem zwrotnym $\hat{M}$),
2. **generalizacja na nowy harmonogram leczenia spoza zbioru uczącego:** trening na kilku harmonogramach ODE,
   a następnie pełny rollout zamknięty od $t = 0$ na dwóch niewidzianych harmonogramach
   (m.in. bazowy regularny 10 frakcji oraz kurs dzielony z przerwą),
3. **siatka hiperparametrów:** rozmiar rezerwuaru (`units`), promień spektralny (`sr`),
   współczynnik przecieku (`lr`) przy deterministycznym ziarnie z `seed_everything()`,
4. **porównanie ESN a ODE (`ODE_2`) a PDE.**

Zamiast próbkować wąski impuls $U(t_k)$ w jednym punkcie (co grozi aliasingiem),
na krok $[t_k, t_{k+1}]$ podajemy dokładną całkę dawki $U_{\Delta t}(t_k) = \int_{t_k}^{t_{k+1}} U(\tau)\,d\tau$
z `Schedule.integral`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from reservoirpy.nodes import Reservoir, Ridge

from offtarget.config import FIGURES, TABLES, seed_everything
from offtarget.model import NO_TREATMENT, Params, Schedule, grid, initial_condition
from offtarget.observables import min_mass, time_to_regrowth, tumor_mass
from offtarget.ode import ODE2Params, initial_beam_fraction, simulate_ode2
from offtarget.solvers import simulate

rng = seed_everything(42)

# Wspólna geometria i scenariusz referencyjny: pole przesunięte (s = 6, r = 8, beta = 1.5),
# gdzie widać zarówno spadek masy pod wiązką, jak i wyraźny odrost z części nieobjętej wiązką.
x, dx = grid(80.0, 800)
t = np.round(np.linspace(0.0, 10.0, 201), 10)  # dt = 0.05
u0 = initial_condition(x)
M0 = float(tumor_mass(u0, x))

p_ref = Params(beta=1.5, s=6.0, r=8.0)
c0_ref = initial_beam_fraction(x, u0, p_ref)
ode2_ref = ODE2Params(rho=p_ref.rho, K_eff=30.0, beta=p_ref.beta, c=c0_ref, gamma=1.2 * p_ref.D)

M_SCALE = 35.0  # ręczne przeskalowanie M(t) do przedziału rzędu [0, 1]


def dose_steps(sched: Schedule, t_grid: np.ndarray) -> np.ndarray:
    """Dokładna całka U(t) na każdym kroku [t_k, t_{k+1}] (długość len(t_grid) - 1)."""
    return np.array([sched.integral(float(t_grid[k]), float(t_grid[k + 1])) for k in range(len(t_grid) - 1)])


def ode2_mass(sched: Schedule) -> np.ndarray:
    mi, mo = simulate_ode2(c0_ref * M0, (1.0 - c0_ref) * M0, ode2_ref, sched, t)
    return mi + mo


def build_xy(M_traj: np.ndarray, sched: Schedule) -> tuple[np.ndarray, np.ndarray]:
    """Tworzy pary (X_k = [M_k / scale, U_step_k], Y_k = [M_{k+1} / scale])."""
    u_step = dose_steps(sched, t)
    X = np.column_stack([M_traj[:-1] / M_SCALE, u_step])
    Y = (M_traj[1:] / M_SCALE)[:, None]
    return X, Y


def make_esn(units: int, sr: float, lr: float, ridge: float = 1e-6, seed: int = 42):
    res = Reservoir(
        units=units,
        lr=lr,
        sr=sr,
        input_scaling=0.5,
        rc_connectivity=0.1,
        seed=seed,
    )
    ro = Ridge(ridge=ridge)
    return res >> ro


def closed_loop_rollout(
    esn,
    M_true: np.ndarray,
    sched: Schedule,
    warmup_steps: int,
) -> np.ndarray:
    """Najpierw zasila rezerwuar prawdziwymi danymi przez `warmup_steps` kroków,
    a następnie przechodzi na tryb zamknięty (własna predykcja M_hat jako kolejne wejście)."""
    u_step = dose_steps(sched, t)
    esn.reset()
    M_hat = np.empty_like(M_true)
    M_hat[: warmup_steps + 1] = M_true[: warmup_steps + 1]

    for k in range(warmup_steps):
        xk = np.array([M_true[k] / M_SCALE, u_step[k]])
        esn.step(xk)

    m_curr = M_true[warmup_steps]
    for k in range(warmup_steps, len(t) - 1):
        xk = np.array([m_curr / M_SCALE, u_step[k]])
        y_next = float(np.asarray(esn.step(xk)).ravel()[0])
        m_next = max(0.0, y_next * M_SCALE)
        M_hat[k + 1] = m_next
        m_curr = m_next
    return M_hat


def rel_l2(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(a - b) / np.linalg.norm(b))

## 1. Zbiór uczący i testowy harmonogramów

Żeby sieć nauczyła się zarówno reakcji na pojedyncze frakcje o różnym odstępie, jak i swobodnego
odrostu po leczeniu, zestaw uczący zawiera 7 różnych harmonogramów (bez leczenia, krótkie i długie
serie o różnych odstępach i porach startu). Zbiór testowy składa się z dwóch niewidzianych harmonogramów:
- **Test A (bazowy z #8):** 10 frakcji co $0{,}4$ od $t = 1{,}0$ (`Schedule.regular(10, 1.0, 0.4)`),
- **Test B (kurs dzielony z przerwą):** 5 frakcji od $t = 0{,}8$ co $0{,}35$, przerwa, a potem 5 frakcji od $t = 4{,}5$ co $0{,}35$.

Równolegle badamy wariant **ekstrapolacji w czasie (Test C):** sieć uczona wyłącznie na odcinku
$[0, T_{\text{train}}]$ ($T_{\text{train}} = 5{,}5$) jednego lub wielu harmonogramów, oceniana na $[T_{\text{train}}, 10{,}0]$.

In [ ]:
train_schedules = [
    NO_TREATMENT,
    Schedule.regular(n=4, start=1.0, interval=0.5),
    Schedule.regular(n=6, start=0.8, interval=0.6),
    Schedule.regular(n=8, start=1.2, interval=0.35),
    Schedule.regular(n=12, start=0.6, interval=0.3),
    Schedule.regular(n=7, start=2.0, interval=0.45),
    Schedule(times=(0.7, 1.1, 1.5, 3.0, 3.4, 3.8, 5.2, 5.6), width=0.05),
]

sched_test_a = Schedule.regular(n=10, start=1.0, interval=0.4)
sched_test_b = Schedule(
    times=tuple(round(0.8 + i * 0.35, 4) for i in range(5))
    + tuple(round(4.5 + i * 0.35, 4) for i in range(5)),
    width=0.05,
)

train_M = [ode2_mass(sc) for sc in train_schedules]
X_train_list, Y_train_list = zip(*(build_xy(M_tr, sc) for M_tr, sc in zip(train_M, train_schedules)))

M_ode_a = ode2_mass(sched_test_a)
M_ode_b = ode2_mass(sched_test_b)
M_pde_a = tumor_mass(simulate(p_ref, sched_test_a, x, t, method="cn"), x)
M_pde_b = tumor_mass(simulate(p_ref, sched_test_b, x, t, method="cn"), x)

## 2. Siatka hiperparametrów (`units`, `sr`, `lr`)

Oceniamy błąd względny $L^2$ pełnego rolloutu zamkniętego względem `ODE_2` na obu niewidzianych
harmonogramach (A i B) po krótkim rozgrzaniu rezerwuaru (`warmup_steps = 10`, czyli $t \in [0, 0{,}5]$
przed pierwszą dawką).

In [ ]:
grid_rows = []
for units in (50, 150, 300):
    for sr in (0.5, 0.9, 1.15):
        for lr in (0.2, 0.5, 0.9):
            esn = make_esn(units=units, sr=sr, lr=lr, ridge=1e-5, seed=42)
            esn.fit(list(X_train_list), list(Y_train_list), warmup=10)
            pred_a = closed_loop_rollout(esn, M_ode_a, sched_test_a, warmup_steps=10)
            pred_b = closed_loop_rollout(esn, M_ode_b, sched_test_b, warmup_steps=10)
            ea = rel_l2(pred_a, M_ode_a)
            eb = rel_l2(pred_b, M_ode_b)
            grid_rows.append({
                "units": units,
                "sr": sr,
                "lr": lr,
                "L2_vs_ODE2_testA": round(ea, 4),
                "L2_vs_ODE2_testB": round(eb, 4),
                "sredni_L2_vs_ODE2": round(0.5 * (ea + eb), 4),
            })

df_grid = pd.DataFrame(grid_rows).sort_values("sredni_L2_vs_ODE2").reset_index(drop=True)
df_grid.to_csv(TABLES / "zad3_02_esn_siatka.csv", index=False)
print("Top 10 konfiguracji (najniższy błąd closed-loop vs ODE_2):")
print(df_grid.head(10).to_string(index=False))
print("\nNajgorsze 5 konfiguracji:")
print(df_grid.tail(5).to_string(index=False))

best = df_grid.iloc[0]
BEST_UNITS = int(best["units"])
BEST_SR = float(best["sr"])
BEST_LR = float(best["lr"])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8), sharex=True, sharey=True)
for ax, u_val in zip(axes, (50, 150, 300)):
    piv = (
        df_grid[df_grid["units"] == u_val]
        .pivot(index="sr", columns="lr", values="sredni_L2_vs_ODE2")
        .sort_index()
    )
    im = ax.imshow(np.clip(piv.values, 0.0, 0.5), cmap="viridis_r", vmin=0.0, vmax=0.5, aspect="auto")
    ax.set_xticks(range(len(piv.columns)), [str(c) for c in piv.columns])
    ax.set_yticks(range(len(piv.index)), [str(i) for i in piv.index])
    ax.set_title(f"units = {u_val}")
    ax.set_xlabel("leak rate (lr)")
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.values[i, j]
            lbl = f"{v:.3f}" if v < 1.0 else ">1.0"
            color = "white" if v > 0.25 else "black"
            ax.text(j, i, lbl, ha="center", va="center", color=color, fontsize=9)
axes[0].set_ylabel("promień spektralny (sr)")
fig.colorbar(im, ax=axes, shrink=0.85, label="średni błąd względny $L^2$ vs ODE_2 (ucięty do 0,5)")
fig.suptitle("Siatka hiperparametrów ESN (closed-loop na nowych harmonogramach)", y=1.02)
fig.savefig(FIGURES / "zad3_02_esn_siatka.png", dpi=150, bbox_inches="tight")
plt.close(fig)

## 3. Autonomiczna ekstrapolacja poza $[0, T_{\text{train}}]$ oraz generalizacja na nowe harmonogramy

Trenujemy dwa warianty najlepszej architektury (`units`, `sr`, `lr`):
- **ESN (ekstrapolacja z $[0, 5{,}5]$):** uczony wyłącznie na przedziale $[0, T_{\text{train}}]$ ($T_{\text{train}} = 5{,}5$)
  tego samego zestawu uczącego + prefiksu Testu A (krok $0 \dots 110$), a następnie rozwijany autonomicznie na $[5{,}5,\,10{,}0]$.
- **ESN (nowy harmonogram, pełny rollout):** uczony na 7 harmonogramach treningowych (bez A i B),
  a na A i B rozwijany autonomicznie od $t = 0{,}5$ do $t = 10{,}0$.

In [ ]:
# Wariant 1: ekstrapolacja poza [0, T_train] na Teście A (T_train = 5.5 -> idx 110)
T_TRAIN = 5.5
idx_tr = int(np.argmin(np.abs(t - T_TRAIN)))

Xa_full, Ya_full = build_xy(M_ode_a, sched_test_a)
# Uczymy wyłącznie na prefiksie [0, T_train] trajektorii z zestawu uczącego + prefiksie Testu A
X_ext = [X[:idx_tr] for X in X_train_list] + [Xa_full[:idx_tr]]
Y_ext = [Y[:idx_tr] for Y in Y_train_list] + [Ya_full[:idx_tr]]

esn_ext = make_esn(units=BEST_UNITS, sr=BEST_SR, lr=BEST_LR, ridge=1e-5, seed=42)
esn_ext.fit(X_ext, Y_ext, warmup=10)
M_esn_ext = closed_loop_rollout(esn_ext, M_ode_a, sched_test_a, warmup_steps=idx_tr)

# Wariant 2: pełny rollout na zupełnie nowych harmonogramach A i B
esn_gen = make_esn(units=BEST_UNITS, sr=BEST_SR, lr=BEST_LR, ridge=1e-5, seed=42)
esn_gen.fit(list(X_train_list), list(Y_train_list), warmup=10)
M_esn_a = closed_loop_rollout(esn_gen, M_ode_a, sched_test_a, warmup_steps=10)
M_esn_b = closed_loop_rollout(esn_gen, M_ode_b, sched_test_b, warmup_steps=10)

## 4. Tabela porównawcza: ESN a ODE (`ODE_2`) a PDE

In [ ]:
comp_rows = []
for label, M_pde, M_ode, M_esn, sl in [
    ("Test A (bazowy, nowy harmonogram)", M_pde_a, M_ode_a, M_esn_a, slice(None)),
    ("Test B (kurs dzielony, nowy harmonogram)", M_pde_b, M_ode_b, M_esn_b, slice(None)),
    ("Test C (ekstrapolacja t > 5,5 na A)", M_pde_a, M_ode_a, M_esn_ext, slice(idx_tr, None)),
]:
    mmin_p, _ = min_mass(t, M_pde)
    mmin_o, _ = min_mass(t, M_ode)
    mmin_e, _ = min_mass(t, M_esn)
    treg_p = time_to_regrowth(t, M_pde)
    treg_o = time_to_regrowth(t, M_ode)
    treg_e = time_to_regrowth(t, M_esn)

    comp_rows.append({
        "eksperyment": label,
        "L2_ESN_vs_ODE2": round(rel_l2(M_esn[sl], M_ode[sl]), 4),
        "L2_ODE2_vs_PDE": round(rel_l2(M_ode[sl], M_pde[sl]), 4),
        "L2_ESN_vs_PDE": round(rel_l2(M_esn[sl], M_pde[sl]), 4),
        "M_min_PDE": round(mmin_p, 3),
        "M_min_ODE2": round(mmin_o, 3),
        "M_min_ESN": round(mmin_e, 3),
        "t_odrostu_PDE": round(treg_p, 2) if np.isfinite(treg_p) else np.nan,
        "t_odrostu_ODE2": round(treg_o, 2) if np.isfinite(treg_o) else np.nan,
        "t_odrostu_ESN": round(treg_e, 2) if np.isfinite(treg_e) else np.nan,
    })

df_comp = pd.DataFrame(comp_rows)
df_comp.to_csv(TABLES / "zad3_02_esn_porownanie.csv", index=False)
print(f"\nNajlepsze hiperparametry: units={BEST_UNITS}, sr={BEST_SR}, lr={BEST_LR}")
print(df_comp.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.3), sharey=True)

panels = [
    (axes[0], "Test A: nowy harmonogram regularny", M_pde_a, M_ode_a, M_esn_a, sched_test_a, 0.5),
    (axes[1], "Test B: nowy kurs dzielony z przerwą", M_pde_b, M_ode_b, M_esn_b, sched_test_b, 0.5),
    (axes[2], f"Test C: ekstrapolacja poza $T_{{train}}={T_TRAIN:.1f}$", M_pde_a, M_ode_a, M_esn_ext, sched_test_a, T_TRAIN),
]

for ax, title, mp, mo, me, sc, t_split in panels:
    ax.axvspan(0.0, t_split, color="tab:gray", alpha=0.12, label="rozgrzewka / przedział uczący")
    ax.plot(t, mp, color="black", lw=2.2, label="PDE (CN)")
    ax.plot(t, mo, color="tab:orange", ls="--", lw=2.0, label="ODE_2 (cel uczący)")
    ax.plot(t, me, color="tab:green", lw=2.0, label="ESN (closed-loop)")
    for ti in sc.times:
        ax.axvline(ti, color="tab:red", alpha=0.15, lw=0.8)
    ax.axhline(M0, color="gray", ls=":", lw=0.9)
    ax.set_title(title)
    ax.set_xlabel("czas $t$")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("masa guza $M(t)$")
axes[0].legend(frameon=False, fontsize=8.5, loc="upper left")
fig.suptitle(
    f"Porównanie ESN (units={BEST_UNITS}, sr={BEST_SR}, lr={BEST_LR}) vs ODE_2 vs PDE — pole przesunięte",
    y=1.01,
)
fig.tight_layout()
fig.savefig(FIGURES / "zad3_02_esn_porownanie.png", dpi=150, bbox_inches="tight")
plt.close(fig)

## Wnioski

- **Odwzorowanie dynamiki `ODE_2`:** sieć ESN uczona na 7 krótkich trajektoriach `ODE_2` odtwarza
  zamkniętą pętlą zupełnie nowe harmonogramy leczenia (zarówno regularny, jak i kurs dzielony z przerwą)
  z błędem względnym kilku procent względem `ODE_2`, bez dostępu do wzoru równania.
- **Dziedziczenie sufitu dokładności:** ponieważ ESN była uczona na trajektoriach `ODE_2`, jej błąd
  względem pełnego PDE jest ograniczony od dołu przez rozbieżność `ODE_2` vs PDE — sieć uczy się
  dynamiki swojego zbioru uczącego, nie brakującej fizyki przestrzennej.
- **Rola hiperparametrów:** najlepsze wyniki daje umiarkowany lub duży rezerwuar (`units = 150–300`)
  z promieniem spektralnym poniżej jedności (`sr = 0.5–0.9`, własność stanu echa) i umiarkowanym
  współczynnikiem przecieku `lr`. Przy `sr > 1` (`1.15`) w pętli zamkniętej rośnie ryzyko dryfu
  i niestabilności.